# Answer-bearing and grouped retrieval reevaluation

이 노트북은 **Codex coder agent**가 저장된 개발셋 결과만 사용해 fresh kernel에서 실행한다. GPU·모델·custom code·network·API·embedding·Chroma/HNSW 호출은 없다. `ANSWER_BEARING`은 level을 완화한 term-bearing 진단 기준이며 사실적으로 완전한 정답과 동일하지 않다. holdout 및 운영 일반화 결론은 범위 밖이다.


In [1]:
import csv, hashlib, json, math, os, tempfile, unicodedata
from collections import defaultdict
from pathlib import Path

PROJECT_ROOT = Path.cwd().parents[0].resolve()
OUTPUT_ROOT = PROJECT_ROOT / 'notebooks/data/18_answer_bearing_grouped_retrieval_reevaluation'
SOURCE_13 = PROJECT_ROOT / 'notebooks/data/13_hierarchical_chunking_retrieval'
SOURCE_16 = PROJECT_ROOT / 'notebooks/data/16_normalized_rrf_weight_ablation'
SOURCE_17 = PROJECT_ROOT / 'notebooks/data/17_gte_reranker_ablation'
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
TOLERANCE = 1e-12
VIEWS = ('strict_raw', 'answer_bearing_raw', 'strict_exact_doc_dedup', 'answer_bearing_exact_doc_dedup', 'answer_bearing_gold_family_oracle')
COMPARISONS = {f'{weight}_top{depth}': {'weight': weight, 'depth': depth} for weight in ('vector_0.4_bm25_0.6', 'vector_0.5_bm25_0.5') for depth in (20, 50)}
PRIMARY_COMPARISON = 'vector_0.4_bm25_0.6_top50'
GROUPS = ('all', 'card', 'evidence', 'numeric', 'semantic')
EXPECTED_DENOMINATORS = {'all': 30, 'card': 10, 'evidence': 20, 'numeric': 10, 'semantic': 10}
METRICS = ('card_hit_at_3', 'hit_at_3', 'recall_at_5', 'mrr_at_5', 'ndcg_at_5')
NORMALIZATION_CONTRACT = '" ".join(unicodedata.normalize("NFKC", str(value)).lower().split())'
INTERPRETATION_RULES = {
 'level_or_duplicate_effect_strong': 'strict declines while answer-bearing and family do not regress',
 'oracle_only_recovery': 'family recovers while exact dedup does not',
 'query_independent_dedup_candidate': 'exact dedup recovers',
 'answer_chunk_misranking_strengthened': 'answer-bearing also declines',
 'decision_invariance': 'diagnostic results do not automatically change the existing retain_no_reranker decision',
}
EVALUATION_CONTRACT = {
 'normalization': NORMALIZATION_CONTRACT, 'numeric_canonicalization': False, 'punctuation_removal': False, 'morphology': False, 'synonyms': False, 'fuzzy_match': False,
 'strict': 'expected card + expected level + all normalized required terms contained in document',
 'answer_bearing': 'expected card + all normalized required terms contained in document; level ignored',
 'answer_bearing_alias': 'level_relaxed_term_bearing', 'answer_bearing_is_not_factual_completeness': True,
 'views': VIEWS, 'exact_dedup_key': ['card_key', 'SHA256(normalized(document))'], 'exact_dedup_uses_query_gold_or_level': False,
 'gold_family_oracle': 'one query-specific family for all ANSWER_BEARING chunks; first gain only; evaluation oracle, not operational dedup',
 'excluded_grouping': ['content containment', 'fuzzy grouping'], 'groups': EXPECTED_DENOMINATORS, 'primary_comparison': PRIMARY_COMPARISON,
 'ranking': {'no_reranker': '16 fused_rank', 'gte': '17 logit descending, tie original RRF rank, then chunk ID'},
 'interpretation_priority': ['Hit/MRR for raw answer-bearing', 'gold-family Recall/nDCG', 'exact-document dedup', 'raw answer-bearing Recall/nDCG is parent-duplicate sensitive'],
 'interpretation_rules': INTERPRETATION_RULES, 'existing_decision_automatically_changed': False,
}
print({'contract_frozen_before_results': True, 'views': VIEWS, 'comparisons': COMPARISONS, 'primary': PRIMARY_COMPARISON, 'gpu_model_network_api_embedding_chroma_calls': 0})


{'contract_frozen_before_results': True, 'views': ('strict_raw', 'answer_bearing_raw', 'strict_exact_doc_dedup', 'answer_bearing_exact_doc_dedup', 'answer_bearing_gold_family_oracle'), 'comparisons': {'vector_0.4_bm25_0.6_top20': {'weight': 'vector_0.4_bm25_0.6', 'depth': 20}, 'vector_0.4_bm25_0.6_top50': {'weight': 'vector_0.4_bm25_0.6', 'depth': 50}, 'vector_0.5_bm25_0.5_top20': {'weight': 'vector_0.5_bm25_0.5', 'depth': 20}, 'vector_0.5_bm25_0.5_top50': {'weight': 'vector_0.5_bm25_0.5', 'depth': 50}}, 'primary': 'vector_0.4_bm25_0.6_top50', 'gpu_model_network_api_embedding_chroma_calls': 0}


In [2]:
def normalize(value):
    return " ".join(unicodedata.normalize("NFKC", str(value)).lower().split())

def sha256_file(path):
    digest = hashlib.sha256()
    with path.open('rb') as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b''): digest.update(block)
    return digest.hexdigest()

def write_json(path, value):
    with tempfile.NamedTemporaryFile('w', encoding='utf-8', dir=path.parent, delete=False) as handle:
        temporary = Path(handle.name); json.dump(value, handle, ensure_ascii=False, indent=2); handle.write('\n')
    os.replace(temporary, path)

def write_jsonl(path, rows):
    with tempfile.NamedTemporaryFile('w', encoding='utf-8', dir=path.parent, delete=False) as handle:
        temporary = Path(handle.name)
        for row in rows: handle.write(json.dumps(row, ensure_ascii=False, sort_keys=True) + '\n')
    os.replace(temporary, path)

def write_csv(path, rows):
    rows = list(rows); columns = list(dict.fromkeys(key for row in rows for key in row))
    with tempfile.NamedTemporaryFile('w', encoding='utf-8', newline='', dir=path.parent, delete=False) as handle:
        temporary = Path(handle.name); writer = csv.DictWriter(handle, fieldnames=columns); writer.writeheader(); writer.writerows(rows)
    os.replace(temporary, path)

INPUTS = {
 'chunks_13': SOURCE_13 / 'chunks.jsonl', 'queries_13': SOURCE_13 / 'retrieval_per_query.csv',
 'candidates_16': SOURCE_16 / 'rrf_weight_candidates.csv',
 'pair_scores_17': SOURCE_17 / 'gte_reranker_pair_scores.csv', 'per_query_17': SOURCE_17 / 'gte_reranker_per_query.csv', 'summary_17': SOURCE_17 / 'gte_reranker_summary.json',
}
input_hashes_before = {name: sha256_file(path) for name, path in INPUTS.items()}
chunks = [json.loads(line) for line in INPUTS['chunks_13'].read_text(encoding='utf-8').splitlines()]
chunk_by_id = {chunk['id']: chunk for chunk in chunks}
assert len(chunks) == len(chunk_by_id) == 327
query_source = list(csv.DictReader(INPUTS['queries_13'].open(encoding='utf-8', newline='')))
queries = {}
for row in query_source:
    if row['method'] == 'keyword': queries[row['query_id']] = {'query_id': row['query_id'], 'query': row['query'], 'category': row['category'], 'expected_card': row['expected_card'], 'expected_level': row['expected_level'], 'required_terms': json.loads(row['required_terms'])}
assert len(queries) == 30
assert sum(q['expected_level'] == 'card' for q in queries.values()) == 10
assert sum(q['expected_level'] != 'card' for q in queries.values()) == 20
assert sum(q['category'] == 'numeric_condition' for q in queries.values()) == 10
assert sum(q['category'] == 'semantic' for q in queries.values()) == 10

def strict_set(query):
    return {chunk['id'] for chunk in chunks if chunk['metadata']['card_key'] == query['expected_card'] and chunk['metadata']['level'] == query['expected_level'] and all(normalize(term) in normalize(chunk['document']) for term in query['required_terms'])}

def answer_bearing_set(query):
    return {chunk['id'] for chunk in chunks if chunk['metadata']['card_key'] == query['expected_card'] and all(normalize(term) in normalize(chunk['document']) for term in query['required_terms'])}

strict_by_query = {query_id: strict_set(query) for query_id, query in queries.items()}
answer_by_query = {query_id: answer_bearing_set(query) for query_id, query in queries.items()}
assert all(strict_by_query[q] and answer_by_query[q] and strict_by_query[q].issubset(answer_by_query[q]) for q in queries)

group_members = defaultdict(list); normalized_by_group = defaultdict(set); group_by_chunk = {}
for chunk in chunks:
    text = normalize(chunk['document']); document_sha = hashlib.sha256(text.encode()).hexdigest(); key_text = chunk['metadata']['card_key'] + '\0' + document_sha; group_id = hashlib.sha256(key_text.encode()).hexdigest()
    group_by_chunk[chunk['id']] = group_id; group_members[group_id].append(chunk['id']); normalized_by_group[group_id].add(text)
assert all(len(texts) == 1 for texts in normalized_by_group.values())
group_rows = []
for group_id in sorted(group_members):
    members = sorted(group_members[group_id]); first = chunk_by_id[members[0]]
    group_rows.append({'group_id': group_id, 'card_key': first['metadata']['card_key'], 'normalized_document_sha256': hashlib.sha256(next(iter(normalized_by_group[group_id])).encode()).hexdigest(), 'normalized_document': next(iter(normalized_by_group[group_id])), 'member_chunk_ids': members, 'member_levels': sorted({chunk_by_id[i]['metadata']['level'] for i in members}), 'size': len(members)})
strict_groups = {q: {group_by_chunk[i] for i in ids} for q, ids in strict_by_query.items()}
answer_groups = {q: {group_by_chunk[i] for i in ids} for q, ids in answer_by_query.items()}
assert all(strict_groups[q].issubset(answer_groups[q]) for q in queries)
relevance_rows = [{'query_id': q, 'strict_chunk_ids': sorted(strict_by_query[q]), 'answer_bearing_chunk_ids': sorted(answer_by_query[q]), 'level_relaxed_term_bearing_chunk_ids': sorted(answer_by_query[q]), 'strict_exact_group_ids': sorted(strict_groups[q]), 'answer_bearing_exact_group_ids': sorted(answer_groups[q]), 'gold_family_id': f'gold_family:{q}', 'strict_count': len(strict_by_query[q]), 'answer_bearing_count': len(answer_by_query[q]), 'strict_group_count': len(strict_groups[q]), 'answer_bearing_group_count': len(answer_groups[q])} for q in sorted(queries)]
print({'chunks': len(chunks), 'queries': len(queries), 'exact_groups': len(group_rows), 'duplicate_groups': sum(row['size'] > 1 for row in group_rows), 'duplicate_chunks_beyond_first': sum(row['size'] - 1 for row in group_rows), 'strict_subset_answer_bearing': True})


{'chunks': 327, 'queries': 30, 'exact_groups': 286, 'duplicate_groups': 34, 'duplicate_chunks_beyond_first': 41, 'strict_subset_answer_bearing': True}


In [3]:
candidate_rows = list(csv.DictReader(INPUTS['candidates_16'].open(encoding='utf-8', newline='')))
candidate_rows = [row for row in candidate_rows if row['configuration'] in {spec['weight'] for spec in COMPARISONS.values()}]
pair_rows = list(csv.DictReader(INPUTS['pair_scores_17'].open(encoding='utf-8', newline='')))
scores = {(row['query_id'], row['chunk_id']): float(row['reranker_logit']) for row in pair_rows}
assert len(scores) == len(pair_rows) == 1857 and all(math.isfinite(value) for value in scores.values())
rankings = {}
for comparison, spec in COMPARISONS.items():
    for query_id in queries:
        rows = sorted((row for row in candidate_rows if row['configuration'] == spec['weight'] and row['query_id'] == query_id), key=lambda row: int(row['fused_rank']))
        assert len(rows) == 50 and [int(row['fused_rank']) for row in rows] == list(range(1, 51))
        full_ids = [row['chunk_id'] for row in rows]; assert len(set(full_ids)) == 50
        pool = full_ids[:spec['depth']]; assert len(pool) == len(set(pool)) == spec['depth'] and all((query_id, chunk_id) in scores for chunk_id in pool)
        original_rank = {chunk_id: rank for rank, chunk_id in enumerate(pool, 1)}
        gte = sorted(pool, key=lambda chunk_id: (-scores[(query_id, chunk_id)], original_rank[chunk_id], chunk_id))
        assert len(gte) == len(pool) and set(gte) == set(pool)
        rankings[(comparison, 'no_reranker', query_id)] = pool
        rankings[(comparison, 'gte', query_id)] = gte
for weight in {spec['weight'] for spec in COMPARISONS.values()}:
    for query_id in queries: assert rankings[(f'{weight}_top20', 'no_reranker', query_id)] == rankings[(f'{weight}_top50', 'no_reranker', query_id)][:20]

def compressed_group_ranking(raw_ranking):
    seen, result = set(), []
    for chunk_id in raw_ranking:
        group_id = group_by_chunk[chunk_id]
        if group_id not in seen: seen.add(group_id); result.append(group_id)
    assert result == list(dict.fromkeys(group_by_chunk[i] for i in raw_ranking))
    return result

def view_metrics(query_id, raw_ranking, view):
    query = queries[query_id]; card_hit = int(any(chunk_by_id[i]['metadata']['card_key'] == query['expected_card'] for i in raw_ranking[:3]))
    if view == 'strict_raw': units, relevant = raw_ranking, strict_by_query[query_id]
    elif view == 'answer_bearing_raw': units, relevant = raw_ranking, answer_by_query[query_id]
    elif view == 'strict_exact_doc_dedup': units, relevant = compressed_group_ranking(raw_ranking), strict_groups[query_id]
    elif view == 'answer_bearing_exact_doc_dedup': units, relevant = compressed_group_ranking(raw_ranking), answer_groups[query_id]
    else:
        units, relevant = raw_ranking, {f'gold_family:{query_id}'}
    if view == 'answer_bearing_gold_family_oracle':
        seen_family = False; hits = []
        for chunk_id in units[:5]:
            gain = chunk_id in answer_by_query[query_id] and not seen_family; hits.append(gain); seen_family = seen_family or gain
    else: hits = [unit in relevant for unit in units[:5]]
    assert relevant
    first = next((rank for rank, hit in enumerate(hits, 1) if hit), None)
    dcg = sum(hit / math.log2(rank + 1) for rank, hit in enumerate(hits, 1))
    ideal = sum(1 / math.log2(rank + 1) for rank in range(1, min(5, len(relevant)) + 1))
    result = {'card_hit_at_3': card_hit, 'hit_at_3': int(any(hits[:3])), 'recall_at_5': sum(hits) / len(relevant), 'mrr_at_5': 1 / first if first else 0.0, 'ndcg_at_5': dcg / ideal, 'relevant_unit_count': len(relevant), 'ranking_unit_count': len(units)}
    assert all(0 <= result[name] <= 1 for name in METRICS)
    if view == 'answer_bearing_gold_family_oracle': assert result['relevant_unit_count'] == 1 and result['recall_at_5'] in {0.0, 1.0}
    return result

per_query_rows = []
for comparison in COMPARISONS:
    for system in ('no_reranker', 'gte'):
        for query_id, query in queries.items():
            raw = rankings[(comparison, system, query_id)]
            by_view = {view: view_metrics(query_id, raw, view) for view in VIEWS}
            assert len({by_view[view]['card_hit_at_3'] for view in VIEWS}) == 1
            assert by_view['answer_bearing_raw']['hit_at_3'] >= by_view['strict_raw']['hit_at_3'] and by_view['answer_bearing_raw']['mrr_at_5'] + TOLERANCE >= by_view['strict_raw']['mrr_at_5']
            assert by_view['answer_bearing_exact_doc_dedup']['hit_at_3'] >= by_view['strict_exact_doc_dedup']['hit_at_3'] and by_view['answer_bearing_exact_doc_dedup']['mrr_at_5'] + TOLERANCE >= by_view['strict_exact_doc_dedup']['mrr_at_5']
            for view, values in by_view.items():
                per_query_rows.append({'comparison': comparison, 'weight': COMPARISONS[comparison]['weight'], 'depth': COMPARISONS[comparison]['depth'], 'system': system, 'query_id': query_id, 'question_group': 'card' if query['expected_level'] == 'card' else 'evidence', 'category': query['category'], 'view': view, **values, 'top5_chunk_ids': json.dumps(raw[:5], separators=(',', ':')), 'top5_cards': json.dumps([chunk_by_id[i]['metadata']['card_key'] for i in raw[:5]], ensure_ascii=False, separators=(',', ':')), 'top5_levels': json.dumps([chunk_by_id[i]['metadata']['level'] for i in raw[:5]], separators=(',', ':'))})
assert len(per_query_rows) == 4 * 2 * 30 * 5

def in_group(row, group):
    return group == 'all' or (group == 'card' and row['question_group'] == 'card') or (group == 'evidence' and row['question_group'] == 'evidence') or (group == 'numeric' and row['category'] == 'numeric_condition') or (group == 'semantic' and row['category'] == 'semantic')

summary_rows = []
for comparison in COMPARISONS:
    for system in ('no_reranker', 'gte'):
        for view in VIEWS:
            rows = [row for row in per_query_rows if row['comparison'] == comparison and row['system'] == system and row['view'] == view]
            for group in GROUPS:
                selected = [row for row in rows if in_group(row, group)]; assert len(selected) == EXPECTED_DENOMINATORS[group]
                summary_rows.append({'comparison': comparison, 'weight': COMPARISONS[comparison]['weight'], 'depth': COMPARISONS[comparison]['depth'], 'system': system, 'view': view, 'group': group, 'denominator': len(selected), **{name: sum(float(row[name]) for row in selected) / len(selected) for name in METRICS}})
assert len(summary_rows) == 4 * 2 * 5 * 5
print({'rankings': len(rankings), 'per_query_rows': len(per_query_rows), 'summary_rows': len(summary_rows), 'pair_score_coverage_exact': True, 'metric_ranges': True})


{'rankings': 240, 'per_query_rows': 1200, 'summary_rows': 200, 'pair_score_coverage_exact': True, 'metric_ranges': True}

In [4]:
# Reproduce every stored 17 strict_raw result before interpreting new views.
stored_17_rows = list(csv.DictReader(INPUTS['per_query_17'].open(encoding='utf-8', newline='')))
stored_17 = {(row['configuration'], row['query_id']): row for row in stored_17_rows}
strict_rows = [row for row in per_query_rows if row['view'] == 'strict_raw']
for row in strict_rows:
    if row['system'] == 'gte': old_config = row['comparison']
    else: old_config = COMPARISONS[row['comparison']]['weight'] + '_no_reranker'
    old = stored_17[(old_config, row['query_id'])]
    assert json.loads(row['top5_chunk_ids']) == json.loads(old['top5_chunk_ids'])
    mapping = {'hit_at_3': 'strict_evidence_hit_at_3', 'card_hit_at_3': 'card_hit_at_3', 'recall_at_5': 'recall_at_5', 'mrr_at_5': 'mrr_at_5', 'ndcg_at_5': 'ndcg_at_5'}
    assert all(math.isclose(float(row[new]), float(old[old_name]), rel_tol=0, abs_tol=TOLERANCE) for new, old_name in mapping.items())
stored_17_summary = json.loads(INPUTS['summary_17'].read_text(encoding='utf-8'))['results']['summaries']
stored_17_summary = {(row['configuration'], row['question_group']): row for row in stored_17_summary}
group_map = {'all': 'all', 'card': 'card', 'evidence': 'evidence', 'numeric': 'category:numeric_condition', 'semantic': 'category:semantic'}
for row in summary_rows:
    if row['view'] != 'strict_raw': continue
    old_config = row['comparison'] if row['system'] == 'gte' else COMPARISONS[row['comparison']]['weight'] + '_no_reranker'
    old = stored_17_summary[(old_config, group_map[row['group']])]
    mapping = {'hit_at_3': 'strict_evidence_hit_at_3', 'card_hit_at_3': 'card_hit_at_3', 'recall_at_5': 'recall_at_5', 'mrr_at_5': 'mrr_at_5', 'ndcg_at_5': 'ndcg_at_5'}
    assert row['denominator'] == old['denominator'] and all(math.isclose(float(row[new]), float(old[old_name]), rel_tol=0, abs_tol=TOLERANCE) for new, old_name in mapping.items())

paired_rows, changed_rows, paired_summary = [], [], {}
for comparison in COMPARISONS:
    paired_summary[comparison] = {}
    for view in VIEWS:
        base = {row['query_id']: row for row in per_query_rows if row['comparison'] == comparison and row['system'] == 'no_reranker' and row['view'] == view}
        reranked = {row['query_id']: row for row in per_query_rows if row['comparison'] == comparison and row['system'] == 'gte' and row['view'] == view}
        paired_summary[comparison][view] = {}
        for query_id in queries:
            row = {'comparison': comparison, 'primary': int(comparison == PRIMARY_COMPARISON), 'view': view, 'query_id': query_id, 'question_group': base[query_id]['question_group'], 'category': base[query_id]['category']}
            for metric in METRICS:
                delta = float(reranked[query_id][metric]) - float(base[query_id][metric]); row[f'delta_{metric}'] = delta; row[f'{metric}_outcome'] = 'win' if delta > TOLERANCE else ('loss' if delta < -TOLERANCE else 'tie')
            paired_rows.append(row)
        for group in GROUPS:
            selected = [row for row in paired_rows if row['comparison'] == comparison and row['view'] == view and in_group(row, group)]
            paired_summary[comparison][view][group] = {metric: {'wins': sum(row[f'{metric}_outcome'] == 'win' for row in selected), 'losses': sum(row[f'{metric}_outcome'] == 'loss' for row in selected), 'ties': sum(row[f'{metric}_outcome'] == 'tie' for row in selected), 'mean_delta': sum(float(row[f'delta_{metric}']) for row in selected) / len(selected)} for metric in METRICS}
            assert all(sum(paired_summary[comparison][view][group][metric][name] for name in ('wins', 'losses', 'ties')) == EXPECTED_DENOMINATORS[group] for metric in METRICS)
    for query_id in queries:
        base_ids = rankings[(comparison, 'no_reranker', query_id)][:5]; gte_ids = rankings[(comparison, 'gte', query_id)][:5]
        if base_ids != gte_ids:
            deltas = {view: {metric: next(row[f'delta_{metric}'] for row in paired_rows if row['comparison'] == comparison and row['view'] == view and row['query_id'] == query_id) for metric in METRICS} for view in VIEWS}
            changed_rows.append({'comparison': comparison, 'primary': int(comparison == PRIMARY_COMPARISON), 'query_id': query_id, 'question_group': 'card' if queries[query_id]['expected_level'] == 'card' else 'evidence', 'category': queries[query_id]['category'], 'baseline_top5_chunk_ids': json.dumps(base_ids, separators=(',', ':')), 'gte_top5_chunk_ids': json.dumps(gte_ids, separators=(',', ':')), 'metric_deltas_by_view': json.dumps(deltas, sort_keys=True, separators=(',', ':'))})

coverage_rows = []
for comparison, spec in COMPARISONS.items():
    for query_id, query in queries.items():
        pool = set(rankings[(comparison, 'no_reranker', query_id)]); strict = strict_by_query[query_id]; answer = answer_by_query[query_id]
        coverage_rows.append({'comparison': comparison, 'weight': spec['weight'], 'depth': spec['depth'], 'query_id': query_id, 'strict_relevant_count': len(strict), 'answer_bearing_relevant_count': len(answer), 'strict_candidate_hit': int(bool(pool & strict)), 'strict_candidate_recall': len(pool & strict) / len(strict), 'answer_bearing_candidate_hit': int(bool(pool & answer)), 'answer_bearing_candidate_recall': len(pool & answer) / len(answer), 'gold_family_candidate_recall': float(bool(pool & answer)), 'missing_strict_ids': json.dumps(sorted(strict - pool), separators=(',', ':')), 'missing_answer_bearing_ids': json.dumps(sorted(answer - pool), separators=(',', ':'))})

summary_lookup = {(row['comparison'], row['system'], row['view'], row['group']): row for row in summary_rows}
primary_delta = {view: {metric: summary_lookup[(PRIMARY_COMPARISON, 'gte', view, 'evidence')][metric] - summary_lookup[(PRIMARY_COMPARISON, 'no_reranker', view, 'evidence')][metric] for metric in METRICS} for view in VIEWS}
strict_down = primary_delta['strict_raw']['hit_at_3'] < -TOLERANCE or primary_delta['strict_raw']['mrr_at_5'] < -TOLERANCE
answer_nonregress = primary_delta['answer_bearing_raw']['hit_at_3'] >= -TOLERANCE and primary_delta['answer_bearing_raw']['mrr_at_5'] >= -TOLERANCE
family_nonregress = primary_delta['answer_bearing_gold_family_oracle']['hit_at_3'] >= -TOLERANCE and primary_delta['answer_bearing_gold_family_oracle']['mrr_at_5'] >= -TOLERANCE
exact_nonregress = primary_delta['answer_bearing_exact_doc_dedup']['hit_at_3'] >= -TOLERANCE and primary_delta['answer_bearing_exact_doc_dedup']['mrr_at_5'] >= -TOLERANCE
if strict_down and answer_nonregress and family_nonregress: diagnostic = 'level_or_duplicate_effect_strong'
elif family_nonregress and not exact_nonregress: diagnostic = 'oracle_only_recovery'
elif exact_nonregress and strict_down: diagnostic = 'query_independent_dedup_candidate'
elif primary_delta['answer_bearing_raw']['hit_at_3'] < -TOLERANCE or primary_delta['answer_bearing_raw']['mrr_at_5'] < -TOLERANCE: diagnostic = 'answer_chunk_misranking_strengthened'
else: diagnostic = 'mixed_or_inconclusive'
print({'strict_raw_reproduction_17_exact': True, 'paired_rows': len(paired_rows), 'changed_rows': len(changed_rows), 'primary_evidence_delta': primary_delta, 'diagnostic': diagnostic})


{'strict_raw_reproduction_17_exact': True, 'paired_rows': 600, 'changed_rows': 115, 'primary_evidence_delta': {'strict_raw': {'card_hit_at_3': 0.04999999999999993, 'hit_at_3': -0.19999999999999996, 'recall_at_5': -0.03749999999999998, 'mrr_at_5': -0.11916666666666664, 'ndcg_at_5': -0.10119727802146361}, 'answer_bearing_raw': {'card_hit_at_3': 0.04999999999999993, 'hit_at_3': 0.0, 'recall_at_5': 0.03069444444444447, 'mrr_at_5': -0.02750000000000008, 'ndcg_at_5': 0.013894321588351799}, 'strict_exact_doc_dedup': {'card_hit_at_3': 0.04999999999999993, 'hit_at_3': -0.09999999999999998, 'recall_at_5': -0.012499999999999956, 'mrr_at_5': -0.17333333333333334, 'ndcg_at_5': -0.1248275818985396}, 'answer_bearing_exact_doc_dedup': {'card_hit_at_3': 0.04999999999999993, 'hit_at_3': 0.0, 'recall_at_5': 0.04339285714285723, 'mrr_at_5': -0.025000000000000022, 'ndcg_at_5': 0.021538901329168803}, 'answer_bearing_gold_family_oracle': {'card_hit_at_3': 0.04999999999999993, 'hit_at_3': 0.0, 'recall_at_5': 

In [5]:
write_json(OUTPUT_ROOT / 'evaluation_contract.json', EVALUATION_CONTRACT)
write_jsonl(OUTPUT_ROOT / 'relevance_sets.jsonl', relevance_rows)
write_jsonl(OUTPUT_ROOT / 'exact_document_groups.jsonl', group_rows)
write_csv(OUTPUT_ROOT / 'per_query_metrics.csv', per_query_rows)
write_csv(OUTPUT_ROOT / 'summary.csv', summary_rows)
write_csv(OUTPUT_ROOT / 'paired_deltas.csv', paired_rows)
write_csv(OUTPUT_ROOT / 'changed_queries.csv', changed_rows)
write_csv(OUTPUT_ROOT / 'candidate_coverage.csv', coverage_rows)

summary_json = {'schema_version': 'answer_bearing_grouped_reevaluation_v1', 'scope': {'single_run': True, 'development_only': True, 'queries': 30, 'chunks': 327, 'holdout_used': False}, 'contract': EVALUATION_CONTRACT, 'results': {'summary_rows': summary_rows, 'paired': paired_summary, 'primary_evidence_delta': primary_delta, 'changed_query_rows': len(changed_rows), 'primary_changed_queries': sum(int(row['primary']) for row in changed_rows), 'exact_groups': len(group_rows), 'duplicate_groups': sum(row['size'] > 1 for row in group_rows), 'duplicate_chunks_beyond_first': sum(row['size'] - 1 for row in group_rows)}, 'diagnosis': {'rule': diagnostic, 'explanation': INTERPRETATION_RULES.get(diagnostic, 'mixed evidence; no automatic decision change'), 'existing_gte_decision': 'retain_no_reranker', 'existing_decision_changed': False}, 'korean_glossary': {'strict_raw': '기대 카드·기대 level·필수 용어를 모두 만족하는 청크 기준 원순위', 'answer_bearing_raw / level_relaxed_term_bearing': '기대 카드와 필수 용어를 만족하되 level은 무시한 진단 기준; 사실적 완전 정답과 동일하지 않음', 'exact_doc_dedup': '카드와 정규화 문서가 정확히 같은 청크만 하나로 묶는 방식', 'gold_family_oracle': '질의별 정답 포함 청크를 한 가족으로 보는 평가용 상한이며 운영 중복 제거가 아님', 'Card Hit@3': '원순위 상위 3개에 기대 카드가 있는 비율', 'Hit@3': '해당 view의 관련 단위가 상위 3개에 있는 비율', 'Recall@5': '관련 단위 중 상위 5개가 회수한 비율', 'MRR@5': '첫 관련 단위 순위의 역수 평균', 'nDCG@5': '관련 단위를 상위에 배치한 정도', 'win/loss/tie': '질의별 no-reranker 대비 개선/하락/동률'}, 'interpretation': ['raw answer-bearing Recall/nDCG는 parent 중복 분모에 민감하므로 Hit/MRR을 우선 본다.', '그다음 gold-family Recall/nDCG와 exact-document dedup을 본다.'], 'limitations': ['개발 질의 30개의 single run 진단이며 holdout·운영 일반화 결론이 아니다.', 'answer-bearing은 level을 완화한 용어 포함 기준일 뿐 사실적으로 완전한 정답 판정이 아니다.', 'gold-family는 gold를 사용하는 평가 oracle이며 운영 grouping이 아니다.', 'content-containment와 fuzzy grouping은 평가하지 않았다.', '이번 결과는 기존 GTE retain_no_reranker 판정을 자동 변경하지 않는다.'], 'execution': {'environment': 'skn25', 'fresh_kernel': True, 'gpu_calls': 0, 'model_calls': 0, 'custom_code_calls': 0, 'network_calls': 0, 'api_calls': 0, 'new_embeddings': 0, 'package_installs': 0, 'chroma_hnsw_queries': 0}}
write_json(OUTPUT_ROOT / 'summary.json', summary_json)

readme = f'''# Answer-bearing / grouped retrieval reevaluation

개발 질의 30개의 single-run 진단 결과다. holdout은 사용하지 않았고 운영 일반화를 주장하지 않는다. 저장된 16번 후보와 17번 GTE logit만 사용했다.

## 쉬운 용어 풀이

- strict_raw: 기대 카드·level·필수 용어를 모두 만족하는 기존 엄격 기준
- answer_bearing / level_relaxed_term_bearing: 카드와 필수 용어는 맞지만 level을 무시한 진단 기준. 사실적으로 완전한 정답과 같은 뜻이 아니다.
- exact_doc_dedup: 카드와 정규화 문서가 정확히 같은 청크만 하나로 묶는다.
- gold_family_oracle: 모든 answer-bearing 정답을 한 가족으로 보는 평가용 oracle이며 운영 dedup이 아니다.
- Card Hit@3: 상위 3개에 기대 카드가 있는 비율
- Hit@3: 각 view의 관련 단위가 상위 3개에 있는 비율
- Recall@5: 관련 단위 중 상위 5개가 회수한 비율
- MRR@5: 첫 관련 단위가 앞에 있을수록 높은 값
- nDCG@5: 관련 단위를 상위에 둔 정도
- win/loss/tie: 질의별 no-reranker 대비 개선/하락/동률

raw answer-bearing Recall/nDCG는 중복 분모에 민감하므로 Hit/MRR, gold-family Recall/nDCG, exact dedup 순으로 해석한다. 진단은 `{diagnostic}`이며 기존 `retain_no_reranker` 판정은 바꾸지 않는다.
'''
(OUTPUT_ROOT / 'README.md').write_text(readme, encoding='utf-8')

input_hashes_after = {name: sha256_file(path) for name, path in INPUTS.items()}
assert input_hashes_after == input_hashes_before
output_names = ['evaluation_contract.json', 'relevance_sets.jsonl', 'exact_document_groups.jsonl', 'per_query_metrics.csv', 'summary.csv', 'summary.json', 'paired_deltas.csv', 'changed_queries.csv', 'candidate_coverage.csv', 'README.md']
integrity = {'schema_version': 'answer_bearing_grouped_integrity_v1', 'self_hash_excluded': True, 'inputs': {name: {'path': str(path.relative_to(PROJECT_ROOT)), 'sha256_before': input_hashes_before[name], 'sha256_after': input_hashes_after[name], 'unchanged': True} for name, path in INPUTS.items()}, 'outputs': {name: {'path': str((OUTPUT_ROOT / name).relative_to(PROJECT_ROOT)), 'sha256': sha256_file(OUTPUT_ROOT / name), 'bytes': (OUTPUT_ROOT / name).stat().st_size} for name in output_names}, 'notebook': {'path': 'notebooks/18_answer_bearing_grouped_retrieval_reevaluation.ipynb', 'sha256': 'pending_after_nbclient_serialization'}, 'assertions': {'chunk_count': 327, 'query_count': 30, 'category_numeric': 10, 'category_semantic': 10, 'ids_unique': True, 'candidate_depth_rank_unique_contiguous': True, 'pair_finite_coverage_exact': True, 'metric_ranges': True, 'card_hit_view_invariant': True, 'answer_bearing_hit_mrr_dominates_strict': True, 'gold_family_unit_one_binary_recall': True, 'paired_denominators_exact': True, 'dedup_first_representative_deterministic': True, 'strict_raw_17_exact': True, 'input_hashes_unchanged': True}, 'row_counts': {'relevance_sets': len(relevance_rows), 'exact_document_groups': len(group_rows), 'per_query_metrics': len(per_query_rows), 'summary': len(summary_rows), 'paired_deltas': len(paired_rows), 'changed_queries': len(changed_rows), 'candidate_coverage': len(coverage_rows)}, 'execution': summary_json['execution']}
write_json(OUTPUT_ROOT / 'integrity.json', integrity)
print({'outputs_written': len(output_names) + 1, 'input_hashes_unchanged': True, 'rows': integrity['row_counts'], 'diagnostic': diagnostic})


{'outputs_written': 11, 'input_hashes_unchanged': True, 'rows': {'relevance_sets': 30, 'exact_document_groups': 286, 'per_query_metrics': 1200, 'summary': 200, 'paired_deltas': 600, 'changed_queries': 115, 'candidate_coverage': 120}, 'diagnostic': 'answer_chunk_misranking_strengthened'}


In [6]:
def csv_rows(name): return list(csv.DictReader((OUTPUT_ROOT / name).open(encoding='utf-8', newline='')))
stored_relevance = [json.loads(line) for line in (OUTPUT_ROOT / 'relevance_sets.jsonl').read_text(encoding='utf-8').splitlines()]
stored_groups = [json.loads(line) for line in (OUTPUT_ROOT / 'exact_document_groups.jsonl').read_text(encoding='utf-8').splitlines()]
stored_per_query, stored_summary, stored_paired, stored_changed, stored_coverage = map(csv_rows, ('per_query_metrics.csv', 'summary.csv', 'paired_deltas.csv', 'changed_queries.csv', 'candidate_coverage.csv'))
assert len(stored_relevance) == 30 and len(stored_groups) == len(group_rows)
assert (len(stored_per_query), len(stored_summary), len(stored_paired), len(stored_changed), len(stored_coverage)) == (1200, 200, 600, len(changed_rows), 120)
assert all(math.isfinite(float(row[name])) and 0 <= float(row[name]) <= 1 for row in stored_per_query for name in METRICS)
assert all(sum(row[f'{metric}_outcome'] == outcome for outcome in ('win', 'loss', 'tie')) == 1 for row in stored_paired for metric in METRICS)
stored_integrity = json.loads((OUTPUT_ROOT / 'integrity.json').read_text(encoding='utf-8'))
assert stored_integrity['self_hash_excluded'] and stored_integrity['assertions']['strict_raw_17_exact']
assert {name: sha256_file(path) for name, path in INPUTS.items()} == input_hashes_before
print({'validation': 'PASS', 'rows': [1200, 200, 600, len(stored_changed), 120], 'strict_raw_17_exact': True, 'all_metric_and_integrity_assertions': True, 'gpu_model_network_api_embedding_chroma_calls': 0})


{'validation': 'PASS', 'rows': [1200, 200, 600, 115, 120], 'strict_raw_17_exact': True, 'all_metric_and_integrity_assertions': True, 'gpu_model_network_api_embedding_chroma_calls': 0}


## Section/benefit-only exploratory follow-up

이 하단 후속 실험은 기존 18번 결과를 본 뒤 승인된 adaptive diagnostic이다. Evidence 20개만 대상으로 원래 RRF Top50에서 `section|benefit` 청크를 먼저 필터한다. `leaf_only_top20`은 필터 후 fused rank 앞 20개, `leaf_available_from_top50`은 원래 Top50 안에서 필터 후 남은 전체를 사용한다. 후자는 Top50이 아니라 가변 K이며, 두 variant 모두 동일 후보 집합 안에서 no-reranker와 저장 GTE logit 순위를 비교한다. 기존 GTE 판정을 자동 변경하지 않는다.


In [7]:
LEAF_LEVELS = {'section', 'benefit'}
LEAF_VARIANTS = {'leaf_only_top20': {'fixed_k': 20, 'description': 'Top50을 leaf로 필터한 뒤 fused rank 앞 20개'}, 'leaf_available_from_top50': {'fixed_k': None, 'description': '원래 Top50 안에서 leaf 필터 후 남은 전체; 가변 K'}}
LEAF_GROUPS = ('evidence', 'numeric', 'semantic')
LEAF_DENOMINATORS = {'evidence': 20, 'numeric': 10, 'semantic': 10}
ORIGINAL_PHASE_NOTEBOOK_SHA = '71d089c09304dc05bad63bd8046bda7256d93420c34a96efe93ea687ca90fa5d'
FROZEN_ORIGINAL_HASHES = {
 'evaluation_contract.json': 'a8302929752950b33f547cb8bef45eb32a951b541324fb264fcb1724babcad13',
 'relevance_sets.jsonl': '6eaf3bc1bd63683efe41d6908226173b6a8aa67c20ff8ffbb70bd8cbd93fca8d',
 'exact_document_groups.jsonl': '30e713f3751c45353bada3e7ddaf44bd5fb553ddbd72e485ab67b22db6f4127d',
 'per_query_metrics.csv': '7cd2aadf9184159700fc64df79892b56fa2aff9046749d4d6a451dd6f5869c08',
 'summary.csv': '53c8729d234576cf0bacd69214ff176d2a5b11863e8ceced2e0016e18e740f72',
 'summary.json': '16b7a6a488280a8c5b87e918a3498b979f53e0ae475b1d14820d9aeb59f35077',
 'paired_deltas.csv': '205ca8f2e0df5d4d1de795e59ba7ad0ed036497d0b4a4e25f64d30e0e9397077',
 'changed_queries.csv': 'd0a6bc695ef2d1c698d8a753b2c67ccc526417f0fec7d828887df052ac1cc5d6',
 'candidate_coverage.csv': 'bff33d933a0587097372d11a447502e683d42a75b84476de67844571685d06e1',
 'integrity.json': '60e9ab0892a4bc346fe25f8578a8828ee51ee884b60d9a2b95700a926d1c4619',
}
FROZEN_BASE_README_HASH = '67f539759ebe7ea69d66b153c721be9041e2e9cd5804ab0c6daf6638a032f317'
# Full reruns regenerate the base integrity placeholder; restore its frozen original-phase notebook SHA byte-for-byte.
base_integrity = json.loads((OUTPUT_ROOT / 'integrity.json').read_text(encoding='utf-8')); base_integrity['notebook']['sha256'] = ORIGINAL_PHASE_NOTEBOOK_SHA; write_json(OUTPUT_ROOT / 'integrity.json', base_integrity)
assert {name: sha256_file(OUTPUT_ROOT / name) for name in FROZEN_ORIGINAL_HASHES} == FROZEN_ORIGINAL_HASHES
assert sha256_file(OUTPUT_ROOT / 'README.md') == FROZEN_BASE_README_HASH
followup_inputs_before = {name: sha256_file(path) for name, path in INPUTS.items()}
evidence_query_ids = sorted(query_id for query_id, query in queries.items() if query['expected_level'] != 'card')
assert len(evidence_query_ids) == 20 and all(queries[q]['expected_level'] in LEAF_LEVELS for q in evidence_query_ids)
assert sum(queries[q]['category'] == 'numeric_condition' for q in evidence_query_ids) == 10 and sum(queries[q]['category'] == 'semantic' for q in evidence_query_ids) == 10
leaf_rankings, available_counts = {}, {}
for weight in ('vector_0.4_bm25_0.6', 'vector_0.5_bm25_0.5'):
    for query_id in evidence_query_ids:
        top50 = rankings[(f'{weight}_top50', 'no_reranker', query_id)]
        available = [chunk_id for chunk_id in top50 if chunk_by_id[chunk_id]['metadata']['level'] in LEAF_LEVELS]
        assert len(available) >= 20 and len(available) == len(set(available)) and all(chunk_by_id[i]['metadata']['level'] in LEAF_LEVELS for i in available)
        available_counts[(weight, query_id)] = len(available)
        for variant, spec in LEAF_VARIANTS.items():
            pool = available[:spec['fixed_k']] if spec['fixed_k'] else available
            assert len(pool) == (20 if spec['fixed_k'] else len(available)) and all((query_id, chunk_id) in scores and math.isfinite(scores[(query_id, chunk_id)]) for chunk_id in pool)
            old_rank = {chunk_id: rank for rank, chunk_id in enumerate(pool, 1)}
            gte = sorted(pool, key=lambda chunk_id: (-scores[(query_id, chunk_id)], old_rank[chunk_id], chunk_id))
            assert set(gte) == set(pool) and len(gte) == len(pool)
            leaf_rankings[(weight, variant, 'no_reranker', query_id)] = pool
            leaf_rankings[(weight, variant, 'gte', query_id)] = gte
available_values = list(available_counts.values())
assert min(available_values) >= 20 and max(available_values) <= 50
print({'followup_contract_frozen': True, 'evidence_queries': len(evidence_query_ids), 'expected_levels': sorted({queries[q]['expected_level'] for q in evidence_query_ids}), 'available_k_min': min(available_values), 'available_k_max': max(available_values), 'available_k_values': sorted(set(available_values)), 'original_18_hashes_exact': True})


{'followup_contract_frozen': True, 'evidence_queries': 20, 'expected_levels': ['benefit', 'section'], 'available_k_min': 34, 'available_k_max': 46, 'available_k_values': [34, 35, 36, 37, 38, 39, 40, 41, 42, 46], 'original_18_hashes_exact': True}


In [8]:
leaf_per_query = []
for weight in ('vector_0.4_bm25_0.6', 'vector_0.5_bm25_0.5'):
    for variant in LEAF_VARIANTS:
        for system in ('no_reranker', 'gte'):
            for query_id in evidence_query_ids:
                raw = leaf_rankings[(weight, variant, system, query_id)]; query = queries[query_id]; by_view = {view: view_metrics(query_id, raw, view) for view in VIEWS}
                assert len({by_view[view]['card_hit_at_3'] for view in VIEWS}) == 1
                assert by_view['answer_bearing_raw']['hit_at_3'] >= by_view['strict_raw']['hit_at_3'] and by_view['answer_bearing_raw']['mrr_at_5'] + TOLERANCE >= by_view['strict_raw']['mrr_at_5']
                for view, values in by_view.items():
                    leaf_per_query.append({'weight': weight, 'variant': variant, 'system': system, 'query_id': query_id, 'category': query['category'], 'view': view, 'candidate_count': len(raw), **values, 'top5_chunk_ids': json.dumps(raw[:5], separators=(',', ':')), 'top5_levels': json.dumps([chunk_by_id[i]['metadata']['level'] for i in raw[:5]], separators=(',', ':'))})
assert len(leaf_per_query) == 2 * 2 * 2 * 20 * 5 and all(0 <= float(row[name]) <= 1 and math.isfinite(float(row[name])) for row in leaf_per_query for name in METRICS)

def leaf_in_group(row, group): return group == 'evidence' or (group == 'numeric' and row['category'] == 'numeric_condition') or (group == 'semantic' and row['category'] == 'semantic')
leaf_summary_rows = []
for weight in ('vector_0.4_bm25_0.6', 'vector_0.5_bm25_0.5'):
    for variant in LEAF_VARIANTS:
        for system in ('no_reranker', 'gte'):
            for view in VIEWS:
                rows = [row for row in leaf_per_query if row['weight'] == weight and row['variant'] == variant and row['system'] == system and row['view'] == view]
                for group in LEAF_GROUPS:
                    selected_rows = [row for row in rows if leaf_in_group(row, group)]; assert len(selected_rows) == LEAF_DENOMINATORS[group]
                    candidate_counts = sorted(int(row['candidate_count']) for row in selected_rows); standard_median = (candidate_counts[(len(candidate_counts) - 1) // 2] + candidate_counts[len(candidate_counts) // 2]) / 2
                    leaf_summary_rows.append({'weight': weight, 'variant': variant, 'system': system, 'view': view, 'group': group, 'denominator': len(selected_rows), 'candidate_count_min': min(candidate_counts), 'candidate_count_median': standard_median, 'candidate_count_max': max(candidate_counts), **{name: sum(float(row[name]) for row in selected_rows) / len(selected_rows) for name in METRICS}})
assert len(leaf_summary_rows) == 2 * 2 * 2 * 5 * 3

leaf_paired, leaf_paired_summary, leaf_changed = [], {}, []
existing_primary_losses = {row['query_id'] for row in paired_rows if row['comparison'] == PRIMARY_COMPARISON and row['view'] == 'strict_raw' and row['question_group'] == 'evidence' and row['mrr_at_5_outcome'] == 'loss'}
assert len(existing_primary_losses) == 7 and 'hyundai_numeric' in existing_primary_losses
def first_rank(ranking, relevant): return next((rank for rank, chunk_id in enumerate(ranking, 1) if chunk_id in relevant), None)
for weight in ('vector_0.4_bm25_0.6', 'vector_0.5_bm25_0.5'):
    leaf_paired_summary[weight] = {}
    for variant in LEAF_VARIANTS:
        leaf_paired_summary[weight][variant] = {}
        for view in VIEWS:
            base = {row['query_id']: row for row in leaf_per_query if row['weight'] == weight and row['variant'] == variant and row['system'] == 'no_reranker' and row['view'] == view}; gte = {row['query_id']: row for row in leaf_per_query if row['weight'] == weight and row['variant'] == variant and row['system'] == 'gte' and row['view'] == view}
            leaf_paired_summary[weight][variant][view] = {}
            for query_id in evidence_query_ids:
                out = {'weight': weight, 'variant': variant, 'view': view, 'query_id': query_id, 'category': queries[query_id]['category']}
                for metric in METRICS:
                    delta = float(gte[query_id][metric]) - float(base[query_id][metric]); out[f'delta_{metric}'] = delta; out[f'{metric}_outcome'] = 'win' if delta > TOLERANCE else ('loss' if delta < -TOLERANCE else 'tie')
                leaf_paired.append(out)
            for group in LEAF_GROUPS:
                chosen = [row for row in leaf_paired if row['weight'] == weight and row['variant'] == variant and row['view'] == view and leaf_in_group(row, group)]
                leaf_paired_summary[weight][variant][view][group] = {metric: {'wins': sum(row[f'{metric}_outcome'] == 'win' for row in chosen), 'losses': sum(row[f'{metric}_outcome'] == 'loss' for row in chosen), 'ties': sum(row[f'{metric}_outcome'] == 'tie' for row in chosen), 'mean_delta': sum(float(row[f'delta_{metric}']) for row in chosen) / len(chosen)} for metric in METRICS}
                assert all(sum(leaf_paired_summary[weight][variant][view][group][metric][x] for x in ('wins', 'losses', 'ties')) == LEAF_DENOMINATORS[group] for metric in METRICS)
        for query_id in evidence_query_ids:
            base_rank = leaf_rankings[(weight, variant, 'no_reranker', query_id)]; gte_rank = leaf_rankings[(weight, variant, 'gte', query_id)]
            if base_rank[:5] != gte_rank[:5]:
                leaf_changed.append({'weight': weight, 'variant': variant, 'query_id': query_id, 'category': queries[query_id]['category'], 'candidate_count': len(base_rank), 'was_unfiltered_primary_strict_mrr_loss': int(query_id in existing_primary_losses), 'is_hyundai_numeric': int(query_id == 'hyundai_numeric'), 'baseline_strict_first_rank': first_rank(base_rank, strict_by_query[query_id]) or '', 'gte_strict_first_rank': first_rank(gte_rank, strict_by_query[query_id]) or '', 'baseline_answer_first_rank': first_rank(base_rank, answer_by_query[query_id]) or '', 'gte_answer_first_rank': first_rank(gte_rank, answer_by_query[query_id]) or '', 'baseline_top5_chunk_ids': json.dumps(base_rank[:5], separators=(',', ':')), 'gte_top5_chunk_ids': json.dumps(gte_rank[:5], separators=(',', ':'))})

leaf_coverage = []
for weight in ('vector_0.4_bm25_0.6', 'vector_0.5_bm25_0.5'):
    for variant in LEAF_VARIANTS:
        for query_id in evidence_query_ids:
            full = set(rankings[(f'{weight}_top50', 'no_reranker', query_id)]); leaf = set(leaf_rankings[(weight, variant, 'no_reranker', query_id)]); strict = strict_by_query[query_id]; answer = answer_by_query[query_id]
            removed = full - leaf; removed_card_or_page = {chunk_id for chunk_id in full if chunk_by_id[chunk_id]['metadata']['level'] not in LEAF_LEVELS}; excluded_leaf_tail = {chunk_id for chunk_id in removed if chunk_by_id[chunk_id]['metadata']['level'] in LEAF_LEVELS}
            assert removed == removed_card_or_page | excluded_leaf_tail and not (removed_card_or_page & excluded_leaf_tail)
            if variant == 'leaf_available_from_top50': assert not excluded_leaf_tail
            leaf_coverage.append({'weight': weight, 'variant': variant, 'query_id': query_id, 'category': queries[query_id]['category'], 'unfiltered_count': len(full), 'leaf_count': len(leaf), 'removed_from_top50_count': len(removed), 'removed_card_or_page_count': len(removed_card_or_page), 'excluded_leaf_tail_count': len(excluded_leaf_tail), 'unfiltered_strict_hit': int(bool(full & strict)), 'leaf_strict_hit': int(bool(leaf & strict)), 'unfiltered_strict_recall': len(full & strict) / len(strict), 'leaf_strict_recall': len(leaf & strict) / len(strict), 'unfiltered_answer_hit': int(bool(full & answer)), 'leaf_answer_hit': int(bool(leaf & answer)), 'unfiltered_answer_recall': len(full & answer) / len(answer), 'leaf_answer_recall': len(leaf & answer) / len(answer), 'missing_leaf_strict_ids': json.dumps(sorted(strict - leaf), separators=(',', ':')), 'missing_leaf_answer_ids': json.dumps(sorted(answer - leaf), separators=(',', ':'))})
print({'leaf_rows': len(leaf_per_query), 'leaf_summary_rows': len(leaf_summary_rows), 'leaf_paired_rows': len(leaf_paired), 'leaf_changed_rows': len(leaf_changed), 'coverage_rows': len(leaf_coverage), 'existing_primary_loss_queries': sorted(existing_primary_losses)})


{'leaf_rows': 800, 'leaf_summary_rows': 120, 'leaf_paired_rows': 400, 'leaf_changed_rows': 80, 'coverage_rows': 80, 'existing_primary_loss_queries': ['hana_numeric', 'hyundai_numeric', 'ibk_numeric', 'kb_numeric', 'kb_semantic', 'shinhan_numeric', 'woori_numeric']}


In [9]:
leaf_lookup = {(row['weight'], row['variant'], row['system'], row['view'], row['group']): row for row in leaf_summary_rows}
unfiltered_reference = json.loads((OUTPUT_ROOT / 'summary.json').read_text(encoding='utf-8'))['results']['primary_evidence_delta']
leaf_deltas = {weight: {variant: {view: {metric: leaf_lookup[(weight, variant, 'gte', view, 'evidence')][metric] - leaf_lookup[(weight, variant, 'no_reranker', view, 'evidence')][metric] for metric in METRICS} for view in VIEWS} for variant in LEAF_VARIANTS} for weight in ('vector_0.4_bm25_0.6', 'vector_0.5_bm25_0.5')}
hyundai_audit = [row for row in leaf_changed if row['query_id'] == 'hyundai_numeric']
leaf_summary_json = {'schema_version': 'leaf_only_exploratory_followup_v1', 'scope': {'adaptive_followup': True, 'development_only': True, 'evidence_queries': 20, 'numeric': 10, 'semantic': 10}, 'contract': {'levels': sorted(LEAF_LEVELS), 'weights': ['vector_0.4_bm25_0.6', 'vector_0.5_bm25_0.5'], 'variants': LEAF_VARIANTS, 'same_candidate_set_comparison': True, 'views': VIEWS, 'ranking': {'no_reranker': 'fused rank', 'gte': 'stored logit descending, original rank, chunk ID'}, 'candidate_count_median': 'standard median; average of the two middle values for even denominators', 'coverage_count_definitions': {'removed_from_top50_count': 'all original Top50 candidates absent from the variant pool', 'removed_card_or_page_count': 'original Top50 candidates removed because level is not section or benefit', 'excluded_leaf_tail_count': 'section/benefit candidates remaining after the fixed20 cutoff; zero for available variant'}, 'existing_gte_decision_automatically_changed': False}, 'candidate_counts': {'available_min': min(available_values), 'available_max': max(available_values), 'available_values': sorted(set(available_values)), 'by_weight_query': {f'{weight}|{query_id}': count for (weight, query_id), count in sorted(available_counts.items())}}, 'results': {'summary_rows': leaf_summary_rows, 'paired': leaf_paired_summary, 'evidence_deltas': leaf_deltas, 'unfiltered_primary_0.4_top50_evidence_delta': unfiltered_reference, 'changed_query_rows': len(leaf_changed), 'existing_unfiltered_primary_strict_mrr_loss_queries': sorted(existing_primary_losses), 'hyundai_numeric_audit': hyundai_audit}, 'interpretation': {'diagnostic_only': True, 'existing_decision': 'retain_no_reranker', 'existing_decision_changed': False, 'comparison_note': 'Recovery is judged relative to unfiltered 0.4 Top50 strict MRR delta -0.1192 and answer-bearing MRR delta -0.0275.'}, 'limitations': ['Leaf-only recovery would not establish broad card-query quality.', 'Removing card/page chunks can lose parent context.', 'Both variants remain bounded by the original stored Top50.', 'This adaptive development diagnostic does not alter the existing GTE decision.'], 'korean_glossary': {'leaf_only_top20': '원래 Top50을 section/benefit으로 필터한 뒤 fused rank 앞 20개', 'leaf_available_from_top50': '원래 Top50 안에서 section/benefit 필터 후 남은 전체를 쓰는 가변 K; Top50 자체가 아님', 'candidate ceiling': '고정 후보 안에서 가능한 관련 청크 회수 상한', 'candidate_count_median': '짝수 개이면 가운데 두 값 평균을 쓰는 표준 중앙값', 'removed_from_top50_count': '원래 Top50 중 최종 variant 후보에 들지 못한 전체 수', 'removed_card_or_page_count': 'section/benefit이 아니어서 제거된 card/page 후보 수', 'excluded_leaf_tail_count': 'leaf이지만 fixed20 뒤라 제외된 후보 수; available에서는 0'}, 'execution': {'environment': 'skn25', 'fresh_full_notebook': True, 'gpu_calls': 0, 'model_calls': 0, 'custom_code_calls': 0, 'network_calls': 0, 'api_calls': 0, 'new_embeddings': 0, 'package_installs': 0, 'chroma_hnsw_queries': 0}}
write_csv(OUTPUT_ROOT / 'leaf_only_per_query_metrics.csv', leaf_per_query)
write_csv(OUTPUT_ROOT / 'leaf_only_summary.csv', leaf_summary_rows)
write_json(OUTPUT_ROOT / 'leaf_only_summary.json', leaf_summary_json)
write_csv(OUTPUT_ROOT / 'leaf_only_paired_deltas.csv', leaf_paired)
write_csv(OUTPUT_ROOT / 'leaf_only_candidate_coverage.csv', leaf_coverage)
write_csv(OUTPUT_ROOT / 'leaf_only_changed_queries.csv', leaf_changed)
base_readme = (OUTPUT_ROOT / 'README.md').read_text(encoding='utf-8'); assert sha256_file(OUTPUT_ROOT / 'README.md') == FROZEN_BASE_README_HASH
followup_readme = f'''

## section/benefit-only exploratory follow-up

기존 결과 확인 뒤 추가한 adaptive development diagnostic이다. Evidence 20개만 사용한다. `leaf_only_top20`은 원래 Top50에서 section/benefit을 남긴 뒤 fused rank 앞 20개이며, `leaf_available_from_top50`은 원래 Top50 안에서 필터 후 남은 전체를 쓰는 가변 K다. 후자는 Top50 자체가 아니다.

- available K 범위: {min(available_values)}~{max(available_values)}
- 비교: 같은 leaf 후보 집합에서 no-reranker와 저장 GTE logit 순위
- coverage count: `removed_from_top50_count`는 최종 후보에서 빠진 전체, `removed_card_or_page_count`는 level 필터 제거분, `excluded_leaf_tail_count`는 fixed20 뒤 leaf tail이다. available의 leaf tail은 0이다.
- `candidate_count_median`은 짝수 개에서 가운데 두 값 평균을 쓰는 표준 중앙값이다.
- 기존 unfiltered 0.4 Top50 evidence delta: strict MRR {unfiltered_reference['strict_raw']['mrr_at_5']:.4f}, answer-bearing MRR {unfiltered_reference['answer_bearing_raw']['mrr_at_5']:.4f}
- GPU/model/custom code/network/API/embedding/install/Chroma 호출: 0

이 결과가 회복을 보여도 card query 품질, parent context 보존 또는 원래 Top50 밖 recall을 증명하지 않으며 기존 `retain_no_reranker` 판정을 변경하지 않는다.
'''
(OUTPUT_ROOT / 'README.md').write_text(base_readme.rstrip() + followup_readme, encoding='utf-8')
assert {name: sha256_file(OUTPUT_ROOT / name) for name in FROZEN_ORIGINAL_HASHES} == FROZEN_ORIGINAL_HASHES
followup_inputs_after = {name: sha256_file(path) for name, path in INPUTS.items()}; assert followup_inputs_after == followup_inputs_before
leaf_output_names = ['leaf_only_per_query_metrics.csv', 'leaf_only_summary.csv', 'leaf_only_summary.json', 'leaf_only_paired_deltas.csv', 'leaf_only_candidate_coverage.csv', 'leaf_only_changed_queries.csv']
leaf_integrity = {'schema_version': 'leaf_only_followup_integrity_v1', 'self_hash_excluded': True, 'source_hashes_before': followup_inputs_before, 'source_hashes_after': followup_inputs_after, 'source_unchanged': True, 'original_18_frozen_hashes_before': FROZEN_ORIGINAL_HASHES, 'original_18_hashes_after': {name: sha256_file(OUTPUT_ROOT / name) for name in FROZEN_ORIGINAL_HASHES}, 'original_18_non_readme_exact': True, 'readme_exception': {'base_sha256': FROZEN_BASE_README_HASH, 'followup_sha256': sha256_file(OUTPUT_ROOT / 'README.md'), 'reason': 'approved follow-up section appended'}, 'outputs': {name: {'sha256': sha256_file(OUTPUT_ROOT / name), 'bytes': (OUTPUT_ROOT / name).stat().st_size} for name in leaf_output_names}, 'notebook': {'sha256': 'pending_after_nbclient_serialization'}, 'row_counts': {'per_query': len(leaf_per_query), 'summary': len(leaf_summary_rows), 'paired': len(leaf_paired), 'coverage': len(leaf_coverage), 'changed': len(leaf_changed)}, 'schema_notes': leaf_summary_json['contract']['coverage_count_definitions'], 'assertions': {'evidence_expected_levels_leaf_only': True, 'fixed20_exact': True, 'available_unique_and_bounded': True, 'candidate_levels_leaf_only': True, 'pair_scores_finite_covered': True, 'same_set_permutation': True, 'metrics_in_range': True, 'paired_denominators_exact': True, 'standard_candidate_count_median': True, 'coverage_count_identity': True, 'available_leaf_tail_zero': True, 'original_primary_exact': True, 'source_hashes_unchanged': True}, 'execution': leaf_summary_json['execution']}
write_json(OUTPUT_ROOT / 'leaf_only_integrity.json', leaf_integrity)
print({'leaf_outputs_written': len(leaf_output_names) + 1, 'original_18_non_readme_exact': True, 'readme_exception_recorded': True, 'source_unchanged': True, 'evidence_deltas': leaf_deltas})


{'leaf_outputs_written': 7, 'original_18_non_readme_exact': True, 'readme_exception_recorded': True, 'source_unchanged': True, 'evidence_deltas': {'vector_0.4_bm25_0.6': {'leaf_only_top20': {'strict_raw': {'card_hit_at_3': 0.0, 'hit_at_3': 0.0, 'recall_at_5': 0.03749999999999998, 'mrr_at_5': -0.019166666666666665, 'ndcg_at_5': -0.003272523918167236}, 'answer_bearing_raw': {'card_hit_at_3': 0.0, 'hit_at_3': 0.0, 'recall_at_5': -0.015438311688311557, 'mrr_at_5': -0.03583333333333327, 'ndcg_at_5': -0.032151886600713775}, 'strict_exact_doc_dedup': {'card_hit_at_3': 0.0, 'hit_at_3': 0.04999999999999993, 'recall_at_5': 0.050000000000000044, 'mrr_at_5': -0.029166666666666785, 'ndcg_at_5': 0.0002349368216724157}, 'answer_bearing_exact_doc_dedup': {'card_hit_at_3': 0.0, 'hit_at_3': 0.04999999999999993, 'recall_at_5': -0.008412698412698372, 'mrr_at_5': -0.023333333333333206, 'ndcg_at_5': -0.021237290104704987}, 'answer_bearing_gold_family_oracle': {'card_hit_at_3': 0.0, 'hit_at_3': 0.0, 'recall_

In [10]:
stored_leaf_per_query = csv_rows('leaf_only_per_query_metrics.csv'); stored_leaf_summary = csv_rows('leaf_only_summary.csv'); stored_leaf_paired = csv_rows('leaf_only_paired_deltas.csv'); stored_leaf_coverage = csv_rows('leaf_only_candidate_coverage.csv'); stored_leaf_changed = csv_rows('leaf_only_changed_queries.csv')
assert (len(stored_leaf_per_query), len(stored_leaf_summary), len(stored_leaf_paired), len(stored_leaf_coverage), len(stored_leaf_changed)) == (800, 120, 400, 80, len(leaf_changed))
assert all(row['view'] in VIEWS and 0 <= float(row['mrr_at_5']) <= 1 for row in stored_leaf_per_query)
assert all(int(row['candidate_count']) == 20 for row in stored_leaf_per_query if row['variant'] == 'leaf_only_top20')
assert all(int(row['leaf_count']) >= 20 and int(row['leaf_count']) <= 50 for row in stored_leaf_coverage)
assert all(int(row['removed_from_top50_count']) == int(row['removed_card_or_page_count']) + int(row['excluded_leaf_tail_count']) == 50 - int(row['leaf_count']) for row in stored_leaf_coverage)
assert all(int(row['excluded_leaf_tail_count']) == 0 for row in stored_leaf_coverage if row['variant'] == 'leaf_available_from_top50')
assert all(float(row['candidate_count_median']) * 2 == int(float(row['candidate_count_median']) * 2) for row in stored_leaf_summary)
assert all(sum(row[f'{metric}_outcome'] == x for x in ('win', 'loss', 'tie')) == 1 for row in stored_leaf_paired for metric in METRICS)
stored_leaf_integrity = json.loads((OUTPUT_ROOT / 'leaf_only_integrity.json').read_text(encoding='utf-8'))
assert stored_leaf_integrity['original_18_non_readme_exact'] and stored_leaf_integrity['source_unchanged'] and stored_leaf_integrity['self_hash_excluded']
assert {name: sha256_file(OUTPUT_ROOT / name) for name in FROZEN_ORIGINAL_HASHES} == FROZEN_ORIGINAL_HASHES
assert {name: sha256_file(path) for name, path in INPUTS.items()} == followup_inputs_before
print({'leaf_followup_validation': 'PASS', 'rows': [800, 120, 400, 80, len(stored_leaf_changed)], 'available_k': [min(available_values), max(available_values)], 'original_18_exact': True, 'gpu_model_network_chroma_calls': 0})


{'leaf_followup_validation': 'PASS', 'rows': [800, 120, 400, 80, 80], 'available_k': [34, 46], 'original_18_exact': True, 'gpu_model_network_chroma_calls': 0}


## Follow-up 3 — gold query-type selective reranker oracle

이 사후 진단은 정답 query type(`numeric_condition`, `semantic`, `proper_noun`)을 알고 있다는 oracle 가정 아래 기존 행을 선택한다. 배포 가능한 router가 아니며 새 점수 혼합, 새 ranking, 모델 실행을 하지 않는다. `numeric_condition`과 `proper_noun`은 같은 config의 no-reranker를 통과시키고, `semantic`만 같은 config의 저장 GTE 결과를 선택한다. 0.4/0.5 × Top20/Top50과 5개 기존 view를 유지하고 leaf-only 결과는 사용하지 않는다.

Primary는 0.4:0.6 Top50 evidence20이다. 같은 config no-reranker 대비 5개 view의 Card Hit@3, Hit@3, Recall@5, MRR@5, nDCG@5가 모두 tolerance 1e-12 안에서 비회귀할 때만 `worth_testing_deployable_router`, 아니면 `insufficient_oracle_signal`로 기록한다. 이는 승격 판정이 아니다.


In [11]:
FOLLOWUP2_NOTEBOOK_SHA256 = '8c120e5f5b39ee09032e3893671f34b8bbe3e4253acd2b20c2023638618698cc'
FROZEN_PRE_ORACLE_HASHES = {
 'evaluation_contract.json': 'a8302929752950b33f547cb8bef45eb32a951b541324fb264fcb1724babcad13',
 'relevance_sets.jsonl': '6eaf3bc1bd63683efe41d6908226173b6a8aa67c20ff8ffbb70bd8cbd93fca8d',
 'exact_document_groups.jsonl': '30e713f3751c45353bada3e7ddaf44bd5fb553ddbd72e485ab67b22db6f4127d',
 'per_query_metrics.csv': '7cd2aadf9184159700fc64df79892b56fa2aff9046749d4d6a451dd6f5869c08',
 'summary.csv': '53c8729d234576cf0bacd69214ff176d2a5b11863e8ceced2e0016e18e740f72',
 'summary.json': '16b7a6a488280a8c5b87e918a3498b979f53e0ae475b1d14820d9aeb59f35077',
 'paired_deltas.csv': '205ca8f2e0df5d4d1de795e59ba7ad0ed036497d0b4a4e25f64d30e0e9397077',
 'changed_queries.csv': 'd0a6bc695ef2d1c698d8a753b2c67ccc526417f0fec7d828887df052ac1cc5d6',
 'candidate_coverage.csv': 'bff33d933a0587097372d11a447502e683d42a75b84476de67844571685d06e1',
 'integrity.json': '60e9ab0892a4bc346fe25f8578a8828ee51ee884b60d9a2b95700a926d1c4619',
 'leaf_only_per_query_metrics.csv': 'ecd71071333c8a97a54d504d828b7504a5a6b7b945503a374286ed863143daca',
 'leaf_only_summary.csv': 'ef0d6ef6c60a04ee74c9b7300f0ce9a49ae35ee6d38ae75014c926266899f494',
 'leaf_only_summary.json': '8544ecb5fb9c6441c1bc81d9fd34d18ac4d572a5cead179403589a49706ff9b5',
 'leaf_only_paired_deltas.csv': 'c766f1799e2042c8098cefe593bb006273b750f1f885509b1a9e3830ebf03000',
 'leaf_only_candidate_coverage.csv': '0ee3230f3adc9c5d28f3b4248790d52c55ec82674c6ad580f3a5a35414647f6e',
 'leaf_only_changed_queries.csv': 'b8f492a4b5650fe0a9ef383043af0aa6851f4c436580675d8d61215c125bbb12',
 'leaf_only_integrity.json': 'e9b28c7c493cc95035de991e97d87b79241406818f9a61c89f990a86fa161536',
}
FROZEN_PRE_ORACLE_README_HASH = '9cae70a6208ee82f3ddecdbbd253918c70e0f5018b818975e724a812d6a04267'
restored_leaf_integrity = json.loads((OUTPUT_ROOT / 'leaf_only_integrity.json').read_text(encoding='utf-8'))
restored_leaf_integrity['notebook']['sha256'] = FOLLOWUP2_NOTEBOOK_SHA256
write_json(OUTPUT_ROOT / 'leaf_only_integrity.json', restored_leaf_integrity)
assert {name: sha256_file(OUTPUT_ROOT / name) for name in FROZEN_PRE_ORACLE_HASHES} == FROZEN_PRE_ORACLE_HASHES
assert sha256_file(OUTPUT_ROOT / 'README.md') == FROZEN_PRE_ORACLE_README_HASH
oracle_inputs_before = {name: sha256_file(path) for name, path in INPUTS.items()}

ORACLE_ROUTE = {'numeric_condition': 'no_reranker', 'semantic': 'gte', 'proper_noun': 'no_reranker'}
ORACLE_REFERENCES = {'same_config_no_reranker': 'no_reranker', 'all_gte': 'gte'}
ORACLE_CONTRACT = {
 'schema_version': 'gold_query_type_selective_oracle_v1', 'post_hoc_gold_label_oracle': True, 'deployable_router': False,
 'configs': list(COMPARISONS), 'primary_comparison': PRIMARY_COMPARISON, 'primary_groups': ['evidence', 'numeric', 'semantic'], 'supplemental_groups': ['card', 'all'],
 'route': ORACLE_ROUTE, 'views': list(VIEWS), 'metrics': list(METRICS), 'tie_tolerance': TOLERANCE,
 'selection': 'exact existing per-query system row and top5 selected by gold category; no scoring, mixing, or reranking',
 'gate': {'reference': 'same_config_no_reranker', 'scope': 'primary evidence20', 'rule': 'all five metrics in all five views non-regressing within tolerance', 'pass': 'worth_testing_deployable_router', 'fail': 'insufficient_oracle_signal', 'promotion': False},
 'execution': {'environment': 'skn25', 'fresh_kernel': True, 'cpu_only': True, 'gpu_calls': 0, 'model_or_custom_code_calls': 0, 'network_api_calls': 0, 'new_embedding_calls': 0, 'chroma_hnsw_queries': 0, 'package_installs': 0},
 'korean_glossary': {'gold query-type oracle': '정답 질의 유형을 미리 안다고 가정한 사후 상한 진단', 'passthrough': '기존 순위를 그대로 통과', 'paired delta': '같은 질의에서 두 방식 지표의 차이', 'WLT': '질의별 승/패/동률 수'},
}
write_json(OUTPUT_ROOT / 'selective_oracle_contract.json', ORACLE_CONTRACT)

source_lookup = {(row['comparison'], row['system'], row['query_id'], row['view']): row for row in per_query_rows}
assert len(source_lookup) == len(per_query_rows) and set(query['category'] for query in queries.values()) == set(ORACLE_ROUTE)
assert {category: sum(query['category'] == category for query in queries.values()) for category in ORACLE_ROUTE} == {'numeric_condition': 10, 'semantic': 10, 'proper_noun': 10}
oracle_per_query = []
for comparison in COMPARISONS:
    for query_id, query in queries.items():
        selected_system = ORACLE_ROUTE[query['category']]
        for view in VIEWS:
            source = source_lookup[(comparison, selected_system, query_id, view)]
            row = {**source, 'system': 'gold_query_type_selective_oracle', 'selected_source_system': selected_system, 'route_category': query['category'], 'post_hoc_oracle': True}
            assert row['top5_chunk_ids'] == source['top5_chunk_ids'] and row['top5_cards'] == source['top5_cards'] and row['top5_levels'] == source['top5_levels']
            oracle_per_query.append(row)
assert len(oracle_per_query) == 4 * 30 * 5 and len({(r['comparison'], r['query_id'], r['view']) for r in oracle_per_query}) == len(oracle_per_query)
for comparison in COMPARISONS:
    routed = [row for row in oracle_per_query if row['comparison'] == comparison and row['view'] == VIEWS[0]]
    assert sum(row['selected_source_system'] == 'no_reranker' for row in routed) == 20 and sum(row['selected_source_system'] == 'gte' for row in routed) == 10

oracle_summary = []
for comparison in COMPARISONS:
    for view in VIEWS:
        rows = [row for row in oracle_per_query if row['comparison'] == comparison and row['view'] == view]
        for group in GROUPS:
            selected = [row for row in rows if in_group(row, group)]; assert len(selected) == EXPECTED_DENOMINATORS[group]
            oracle_summary.append({'comparison': comparison, 'weight': COMPARISONS[comparison]['weight'], 'depth': COMPARISONS[comparison]['depth'], 'system': 'gold_query_type_selective_oracle', 'view': view, 'group': group, 'denominator': len(selected), **{metric: sum(float(row[metric]) for row in selected) / len(selected) for metric in METRICS}})
assert len(oracle_summary) == 4 * 5 * 5

oracle_paired = []
for comparison in COMPARISONS:
    for reference_name, reference_system in ORACLE_REFERENCES.items():
        for query_id in queries:
            for view in VIEWS:
                oracle = next(row for row in oracle_per_query if row['comparison'] == comparison and row['query_id'] == query_id and row['view'] == view)
                reference = source_lookup[(comparison, reference_system, query_id, view)]
                paired = {'comparison': comparison, 'weight': COMPARISONS[comparison]['weight'], 'depth': COMPARISONS[comparison]['depth'], 'reference': reference_name, 'query_id': query_id, 'question_group': oracle['question_group'], 'category': oracle['category'], 'view': view, 'selected_source_system': oracle['selected_source_system']}
                for metric in METRICS:
                    delta = float(oracle[metric]) - float(reference[metric])
                    paired[f'oracle_{metric}'] = float(oracle[metric]); paired[f'reference_{metric}'] = float(reference[metric]); paired[f'{metric}_delta'] = delta; paired[f'{metric}_outcome'] = 'win' if delta > TOLERANCE else ('loss' if delta < -TOLERANCE else 'tie')
                assert oracle['top5_chunk_ids'] == source_lookup[(comparison, oracle['selected_source_system'], query_id, view)]['top5_chunk_ids']
                oracle_paired.append(paired)
assert len(oracle_paired) == 4 * 2 * 30 * 5

oracle_wlt = []
for comparison in COMPARISONS:
    for reference_name in ORACLE_REFERENCES:
        for view in VIEWS:
            for group in GROUPS:
                rows = [row for row in oracle_paired if row['comparison'] == comparison and row['reference'] == reference_name and row['view'] == view and in_group(row, group)]
                assert len(rows) == EXPECTED_DENOMINATORS[group]
                for metric in METRICS:
                    counts = {outcome: sum(row[f'{metric}_outcome'] == outcome for row in rows) for outcome in ('win', 'loss', 'tie')}
                    assert sum(counts.values()) == EXPECTED_DENOMINATORS[group]
                    oracle_wlt.append({'comparison': comparison, 'weight': COMPARISONS[comparison]['weight'], 'depth': COMPARISONS[comparison]['depth'], 'reference': reference_name, 'view': view, 'group': group, 'metric': metric, 'denominator': len(rows), 'wins': counts['win'], 'losses': counts['loss'], 'ties': counts['tie'], 'mean_delta': sum(float(row[f'{metric}_delta']) for row in rows) / len(rows)})
assert len(oracle_wlt) == 4 * 2 * 5 * 5 * 5

primary_gate_rows = [row for row in oracle_wlt if row['comparison'] == PRIMARY_COMPARISON and row['reference'] == 'same_config_no_reranker' and row['group'] == 'evidence']
assert len(primary_gate_rows) == 5 * 5
gate_checks = {row['view'] + ':' + row['metric']: row['mean_delta'] >= -TOLERANCE for row in primary_gate_rows}
gate_passed = all(gate_checks.values())
oracle_decision = 'worth_testing_deployable_router' if gate_passed else 'insufficient_oracle_signal'
primary_summary = [row for row in oracle_summary if row['comparison'] == PRIMARY_COMPARISON and row['group'] in ('evidence', 'numeric', 'semantic')]
primary_deltas = {reference: {view: {metric: next(row['mean_delta'] for row in oracle_wlt if row['comparison'] == PRIMARY_COMPARISON and row['reference'] == reference and row['view'] == view and row['group'] == 'evidence' and row['metric'] == metric) for metric in METRICS} for view in VIEWS} for reference in ORACLE_REFERENCES}
oracle_summary_json = {'contract': ORACLE_CONTRACT, 'decision': oracle_decision, 'gate_passed': gate_passed, 'gate_checks': gate_checks, 'primary_summary': primary_summary, 'primary_evidence_paired_deltas': primary_deltas, 'row_counts': {'per_query': len(oracle_per_query), 'summary': len(oracle_summary), 'paired_deltas': len(oracle_paired), 'wlt': len(oracle_wlt)}, 'limitations': ['post-hoc gold query-type labels are unavailable to a deployable router', 'single development set run only', 'stored Top20/Top50 candidates and logits bound the diagnostic', 'no holdout or production generalization claim', 'no leaf-only result is included']}
write_csv(OUTPUT_ROOT / 'selective_oracle_per_query.csv', oracle_per_query)
write_csv(OUTPUT_ROOT / 'selective_oracle_summary.csv', oracle_summary)
write_json(OUTPUT_ROOT / 'selective_oracle_summary.json', oracle_summary_json)
write_csv(OUTPUT_ROOT / 'selective_oracle_paired_deltas.csv', oracle_paired)
write_csv(OUTPUT_ROOT / 'selective_oracle_wlt.csv', oracle_wlt)
print({'oracle_rows': [len(oracle_per_query), len(oracle_summary), len(oracle_paired), len(oracle_wlt)], 'decision': oracle_decision, 'gate_passed': gate_passed})


{'oracle_rows': [600, 100, 1200, 1000], 'decision': 'worth_testing_deployable_router', 'gate_passed': True}


In [12]:
oracle_readme = f'''

## Follow-up 3 — gold query-type selective reranker oracle

정답 질의 유형을 미리 아는 사후 oracle 진단이다. 실제 서비스에서 쓸 수 있는 router가 아니며, 새 점수·순위·모델 호출 없이 기존 no-reranker/GTE 행과 Top5를 그대로 선택했다. numeric condition(숫자 조건)과 proper noun(고유명사)은 no-reranker passthrough(기존 순위 통과), semantic(의미형)은 GTE를 선택했다.

- config: 0.4/0.5 × Top20/Top50; primary: 0.4 Top50 evidence20
- paired delta: 같은 질의에서 oracle과 기준 방식의 지표 차이
- WLT: 질의별 win/loss/tie(승/패/동률) 수
- 판정: `{oracle_decision}` (gate 통과: {gate_passed}); 승격이 아니라 배포 가능한 router를 별도 시험할 가치가 있는지에 대한 진단
- 실행: skn25 fresh kernel, CPU/offline; GPU/model/custom code/network/API/new embedding/Chroma/package install 0

개발셋 단일 실행이고 gold label을 사용했으며, 저장 후보와 logit 범위를 벗어나지 않는다. holdout·운영 일반화를 주장하지 않는다.
'''
current_readme = (OUTPUT_ROOT / 'README.md').read_text(encoding='utf-8')
assert sha256_file(OUTPUT_ROOT / 'README.md') == FROZEN_PRE_ORACLE_README_HASH
(OUTPUT_ROOT / 'README.md').write_text(current_readme.rstrip() + oracle_readme, encoding='utf-8')
oracle_inputs_after = {name: sha256_file(path) for name, path in INPUTS.items()}; assert oracle_inputs_after == oracle_inputs_before
assert {name: sha256_file(OUTPUT_ROOT / name) for name in FROZEN_PRE_ORACLE_HASHES} == FROZEN_PRE_ORACLE_HASHES
oracle_output_names = ['selective_oracle_contract.json', 'selective_oracle_per_query.csv', 'selective_oracle_summary.csv', 'selective_oracle_summary.json', 'selective_oracle_paired_deltas.csv', 'selective_oracle_wlt.csv']
oracle_integrity = {'schema_version': 'gold_query_type_selective_oracle_integrity_v1', 'self_hash_excluded': True, 'source_hashes_before': oracle_inputs_before, 'source_hashes_after': oracle_inputs_after, 'source_unchanged': True, 'existing_18_frozen_hashes_before': FROZEN_PRE_ORACLE_HASHES, 'existing_18_hashes_after': {name: sha256_file(OUTPUT_ROOT / name) for name in FROZEN_PRE_ORACLE_HASHES}, 'existing_followup_1_2_non_readme_byte_exact': True, 'readme_exception': {'before_sha256': FROZEN_PRE_ORACLE_README_HASH, 'after_sha256': sha256_file(OUTPUT_ROOT / 'README.md'), 'reason': 'approved Follow-up 3 section appended'}, 'outputs': {name: {'sha256': sha256_file(OUTPUT_ROOT / name), 'bytes': (OUTPUT_ROOT / name).stat().st_size} for name in oracle_output_names}, 'notebook': {'sha256': 'pending_after_nbclient_serialization'}, 'row_counts': oracle_summary_json['row_counts'], 'assertions': {'route_exact': True, 'aggregate_denominators_exact': True, 'top5_source_exact': True, 'candidate_passthrough_no_new_ranking': True, 'metrics_in_range': True, 'paired_wlt_denominators_exact': True, 'existing_outputs_unchanged': True, 'source_hashes_unchanged': True}, 'execution': ORACLE_CONTRACT['execution']}
write_json(OUTPUT_ROOT / 'selective_oracle_integrity.json', oracle_integrity)

stored_oracle_per_query = csv_rows('selective_oracle_per_query.csv'); stored_oracle_summary = csv_rows('selective_oracle_summary.csv'); stored_oracle_paired = csv_rows('selective_oracle_paired_deltas.csv'); stored_oracle_wlt = csv_rows('selective_oracle_wlt.csv')
assert (len(stored_oracle_per_query), len(stored_oracle_summary), len(stored_oracle_paired), len(stored_oracle_wlt)) == (600, 100, 1200, 1000)
assert all(0 <= float(row[metric]) <= 1 for row in stored_oracle_per_query for metric in METRICS)
assert all(int(row['wins']) + int(row['losses']) + int(row['ties']) == int(row['denominator']) for row in stored_oracle_wlt)
for row in stored_oracle_per_query:
    source = source_lookup[(row['comparison'], row['selected_source_system'], row['query_id'], row['view'])]
    assert row['top5_chunk_ids'] == source['top5_chunk_ids'] and row['top5_cards'] == source['top5_cards'] and row['top5_levels'] == source['top5_levels']
assert {name: sha256_file(path) for name, path in INPUTS.items()} == oracle_inputs_before
assert {name: sha256_file(OUTPUT_ROOT / name) for name in FROZEN_PRE_ORACLE_HASHES} == FROZEN_PRE_ORACLE_HASHES
stored_oracle_integrity = json.loads((OUTPUT_ROOT / 'selective_oracle_integrity.json').read_text(encoding='utf-8'))
assert stored_oracle_integrity['self_hash_excluded'] and stored_oracle_integrity['source_unchanged'] and stored_oracle_integrity['existing_followup_1_2_non_readme_byte_exact']
print({'selective_oracle_validation': 'PASS', 'rows': [600, 100, 1200, 1000], 'decision': oracle_decision, 'existing_outputs_exact': True, 'gpu_model_network_chroma_calls': 0})


{'selective_oracle_validation': 'PASS', 'rows': [600, 100, 1200, 1000], 'decision': 'worth_testing_deployable_router', 'existing_outputs_exact': True, 'gpu_model_network_chroma_calls': 0}
